# FSS at $t=L^{z}$ with block disorder (rho-per-epsilon scans)

Reads data from `../generators/get_slidding_p_rho_per_ep_block.jl` or
`../generators/get_upper_lower_binary_rho_per_ep_block.jl`: one CSV per (control, L,
block_len) under `stavskya_mc/data/block_rho_per_ep/`.

It does the same collapse as the old `analyze_random_*_rho_per_ep.ipynb` notebooks, using
the same `DataCollapse` class (now in `data_collapse.py`), with two changes suggested in
review 4.3.3:
* **Bootstrap errors.** The quoted errors come from resampling the samples, because
  lmfit's standard errors on this piecewise loss are meaningless.
* **Bound check.** A fit that ends up on a parameter bound is reported as failed.

Caveat: fixing $t=L^{z}$ assumes a power-law $z$, which is one of the things under test.
Read these exponents as effective values.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))          # run from stavskya_mc/block_disorder/analysis
import time_log_tools as tl

plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.2, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 13})

In [ ]:
from data_collapse import DataCollapse

In [ ]:
# ---- parameters (mirror the generator you ran) ------------------------------
MODEL = "slidding_p"            # "slidding_p" or "window_binary"
DATA_ROOT = "../../data/block_rho_per_ep"
L_VALS = [1000, 2000, 4000, 8000, 16000]
BLOCK_LEN = 1
Z_VAL = 1.45
N_IC = 3000
# sliding p: control = p
UPPER_VAL = 0.43
LOWER_DIV = 20
P_GRID = [round(0.465 + 0.002 * i, 6) for i in range(21)]
# upper/lower binary: control = eps_bar = p*eps_u + (1-p)*eps_l
P_VAL = 0.8
EPS_U_GRID = sorted(set([round(0.32 + 0.001 * i, 6) for i in range(22)] + [round(0.328 + 0.0005 * i, 6) for i in range(4)]))
TRANS_WINDOW = (0.461, 0.495)   # control range used in the collapse
PC_GUESS = 0.479
NU_GUESS = 1.5
BETA_GUESS = 0.25
NU_RANGE = (0.5, 3.0)
BETA_RANGE = (0.01, 1.0)
N_BOOT = 50
FIG_DIR = "figs"


In [ ]:
if MODEL == "slidding_p":
    MODEL_DIR = "time_rand_slidding_p"
    sets = [(pv, UPPER_VAL, round(UPPER_VAL / LOWER_DIV, 6), pv) for pv in P_GRID]
else:
    MODEL_DIR = "time_rand_window_binary"
    sets = []
    for u in EPS_U_GRID:
        l = round(u / LOWER_DIV, 6)
        sets.append((round(P_VAL * u + (1 - P_VAL) * l, 6), u, l, P_VAL))
Path(FIG_DIR).mkdir(exist_ok=True)
data, missing = {}, []
for L in L_VALS:
    for control, u, l, pv in sets:
        f = tl.block_rho_per_ep_path(DATA_ROOT, MODEL_DIR, L, u, l, pv, BLOCK_LEN, Z_VAL, N_IC)
        if f.exists():
            data[(control, L)] = 1 - pd.read_csv(f)["rho"]          # activity
        else:
            missing.append(str(f))
print(f"loaded {len(data)} files, {len(missing)} missing")
if missing: print("first missing:", missing[0])
df = pd.DataFrame({"observations": list(data.values())},
                  index=pd.MultiIndex.from_tuples(data.keys(), names=["p", "L"]))

In [ ]:
cmap = plt.colormaps["Oranges"].resampled(len(L_VALS) + 3)
fig, ax = plt.subplots(figsize=(8, 5))
for i, L in enumerate(L_VALS):
    s = df.xs(L, level="L")["observations"].sort_index()
    ax.errorbar(s.index, s.apply(np.mean), s.apply(np.std) / np.sqrt(s.apply(len)), capsize=3, color=cmap(i + 3), label=f"L={L}")
ax.axvspan(*TRANS_WINDOW, color="grey", alpha=0.1)
ax.set_xlabel("control"); ax.set_ylabel(r"$1-\rho$ at $t=L^{z}$"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/{MODEL}_rho_per_ep_bl{BLOCK_LEN}.png", dpi=150)

In [ ]:
def fit(frame):
    dc = DataCollapse(frame, p_="p", L_="L", params={}, p_range=list(TRANS_WINDOW), Lmin=min(L_VALS), Lmax=max(L_VALS))
    res = dc.datacollapse(p_c=PC_GUESS, nu=NU_GUESS, beta=BETA_GUESS, beta_vary=True, p_c_vary=True, nu_vary=True,
                          p_c_range=TRANS_WINDOW, nu_range=NU_RANGE, beta_range=BETA_RANGE)
    return dc, res

dc, res = fit(df)
best = {k: res.params[k].value for k in ("p_c", "nu", "beta")}
bounds = {"p_c": TRANS_WINDOW, "nu": NU_RANGE, "beta": BETA_RANGE}
at_bound = [k for k, (lo, hi) in bounds.items() if min(abs(best[k] - lo), abs(best[k] - hi)) < 1e-3 * (hi - lo)]

rng = np.random.default_rng(0)
boot = []
for _ in range(N_BOOT):
    resampled = df.copy()
    resampled["observations"] = [pd.Series(rng.choice(o.to_numpy(), size=len(o), replace=True)) for o in df["observations"]]
    try:
        _, rb = fit(resampled)
        boot.append({k: rb.params[k].value for k in ("p_c", "nu", "beta")})
    except Exception as e:                      # a failed bootstrap fit is dropped, but counted
        print("bootstrap fit failed:", e)
boot = pd.DataFrame(boot)
table = pd.DataFrame({"best fit": best, "bootstrap mean": boot.mean(), "bootstrap std": boot.std()})
print(f"reduced chi2 = {res.redchi:.3f};  bootstrap fits: {len(boot)}/{N_BOOT}")
if at_bound:
    print("FIT FAILED: parameter(s) at a bound:", at_bound)
print("nu here is nu_perp;  nu_t = z * nu_perp =", round(Z_VAL * best["nu"], 3))
table

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
dc.plot_data_collapse(ax=ax)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/{MODEL}_rho_per_ep_bl{BLOCK_LEN}_collapse.png", dpi=150)